In [6]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv('../data/malicious_phish.csv')
df['label'] = df['type'].apply(lambda x: 0 if x == 'benign' else 1)
print(df.shape)

(651191, 3)


In [7]:
import sys
sys.path.append('../src')
from text_features import url_tokenizer

print(url_tokenizer('http://paypal-secure-login.free.fr/update?id=123'))

['http', 'paypal', 'secure', 'login', 'free', 'fr', 'update', 'id']


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    df['url'], df['label'], test_size=0.2, random_state=42
)

tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(tokenizer=url_tokenizer, token_pattern=None, min_df=3, max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])

print("Entraînement du modèle TF-IDF...")
tfidf_pipeline.fit(X_train, y_train)
print("Terminé")

Entraînement du modèle TF-IDF...
Terminé


In [9]:
y_pred = tfidf_pipeline.predict(X_test)
print(f"Accuracy : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))

Accuracy : 94.16%
              precision    recall  f1-score   support

      Benign       0.94      0.98      0.96     85778
   Malicious       0.95      0.87      0.91     44461

    accuracy                           0.94    130239
   macro avg       0.94      0.92      0.93    130239
weighted avg       0.94      0.94      0.94    130239



In [10]:
joblib.dump(tfidf_pipeline, '../models/tfidf_logreg_model.pkl')
print("Pipeline TF-IDF + LogReg sauvegardé")

Pipeline TF-IDF + LogReg sauvegardé
